In [1]:
import pandas as pd
import sys
print(sys.executable)

from sklearn.preprocessing import OneHotEncoder, StandardScaler

/home/vasil/miniconda3/envs/portfolio/bin/python


In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print(df.shape)

(7043, 21)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [6]:
df['TotalCharges'].dtype

<StringDtype(na_value=nan)>

In [7]:
mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isna()
df.loc[mask, ['customerID', 'tenure', 'TotalCharges']]

,customerID,tenure,TotalCharges
488,4472-LVYGI,0,
753,3115-CZMZD,0,
936,5709-LVOEQ,0,
1082,4367-NUYAO,0,
1340,1371-DWPAZ,0,
3331,7644-OMVMY,0,
3826,3213-VVOLG,0,
4380,2520-SGTTA,0,
5218,2923-ARZLG,0,
6670,4075-WKNIU,0,


In [8]:
#Set TotalCharges to numeric, no values will be zero
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['TotalCharges'].dtype

dtype('float64')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [10]:
## Apply one-hot encoding to categorical features and standard scaling to numerical features
categorical_features = df.select_dtypes(include=['str'])
categorical_features = categorical_features.drop(columns=['customerID', 'Churn'])

one_hot = OneHotEncoder(sparse_output=False)
encoded_categorical = one_hot.fit_transform(categorical_features)
encoded_categorical_df = pd.DataFrame(encoded_categorical, columns=one_hot.get_feature_names_out(categorical_features.columns))


In [11]:
numerical_features = df.select_dtypes(include=['int64', 'float64'])
scaler = StandardScaler()
scaled_numerical = scaler.fit_transform(numerical_features)
scaled_numerical_df = pd.DataFrame(scaled_numerical, columns=numerical_features.columns)
scaled_numerical_df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
0,-0.439916,-1.277445,-1.160323,-0.992611
1,-0.439916,0.066327,-0.259629,-0.172165
2,-0.439916,-1.236724,-0.362660,-0.958066
3,-0.439916,0.514251,-0.746535,-0.193672
4,-0.439916,-1.236724,0.197365,-0.938874


In [12]:
df_conc = pd.concat([encoded_categorical_df, scaled_numerical_df], axis=1)
df_conc.head()

,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,...,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-0.439916,-1.277445,-1.160323,-0.992611
1,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,-0.439916,0.066327,-0.259629,-0.172165
2,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,-0.439916,-1.236724,-0.362660,-0.958066
3,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,-0.439916,0.514251,-0.746535,-0.193672
4,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,-0.439916,-1.236724,0.197365,-0.938874


In [13]:
import sys; 
sys.path.append('..')
print(sys.path)


['/home/vasil/miniconda3/envs/portfolio/lib/python311.zip', '/home/vasil/miniconda3/envs/portfolio/lib/python3.11', '/home/vasil/miniconda3/envs/portfolio/lib/python3.11/lib-dynload', '', '/home/vasil/miniconda3/envs/portfolio/lib/python3.11/site-packages', '..']


In [14]:
from src.data import load_data, build_preprocessor

In [15]:
path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = load_data(path)
# print(df.shape)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [16]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

all_cols = set(df.columns)
listed = set(numeric_features) | set(categorical_features)
expected_leftover = {"customerID", "Churn"}

print("In data but not listed anywhere:", all_cols - listed - expected_leftover)
print("Listed but missing from data:   ", listed - all_cols)

In data but not listed anywhere: set()
Listed but missing from data:    set()


In [17]:
prep = build_preprocessor()
df_dropped = df.drop(columns=['customerID', 'Churn'])
df_preprocessed = prep.fit_transform(df_dropped)
print(df_preprocessed.shape)

(7043, 46)


In [18]:
print("Input columns:", df_dropped.shape[1], "-> output columns:", df_preprocessed.shape[1])

Input columns: 19 -> output columns: 46


In [19]:
from sklearn.model_selection import train_test_split

In [21]:
## Load Train and Split

df = load_data(path)
y = (df['Churn'] == 'Yes').astype(int) 
X = df.drop(columns=['customerID', 'Churn'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("Train Churn rate", y_train.mean())
print("Test Churn rate", y_test.mean())

Train shape: (5634, 19) (5634,)
Test shape: (1409, 19) (1409,)
Train Churn rate 0.2653532126375577
Test Churn rate 0.2654364797728886


In [32]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV


from src.data import build_preprocessor

In [34]:
pipe = Pipeline([
    ("pre", build_preprocessor()),
    ("model", GradientBoostingClassifier(random_state=42)),
])

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__learning_rate": [0.005, 0.01, 0.1, 0.2],
    "model__max_depth": [3, 5, 7]
}

search = GridSearchCV(pipe, 
                      param_grid, cv=5, 
                      scoring='roc_auc', 
                      n_jobs=-1,
                      verbose = 1)

search.fit(X_train, y_train)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


In [37]:
print(f"Best ROC_AUC", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)

Best ROC_AUC 0.8471
Best parameters: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300}


In [27]:
proba_train = pipe.predict_proba(X_train)[:, 1]
proba_test = pipe.predict_proba(X_test)[:, 1]

preds_train = pipe.predict(X_train)
preds_test = pipe.predict(X_test)

In [29]:
print("ROC-AUC (Train):", round(roc_auc_score(y_train, proba_train), 4))
print("ROC-AUC (Test) :", round(roc_auc_score(y_test, proba_test), 4))
print("Accuracy (Train):", round(accuracy_score(y_train, preds_train), 4))
print("Accuracy (Test) :", round(accuracy_score(y_test, preds_test), 4))
print()

print("Confusion matrix (Train):")
print(confusion_matrix(y_train, preds_train))
print("Confusion matrix (Test):")
print(confusion_matrix(y_test, preds_test))


ROC-AUC (Train): 0.8822
ROC-AUC (Test) : 0.8434
Accuracy (Train): 0.8296
Accuracy (Test) : 0.8027

Confusion matrix (Train):
[[3812  327]
 [ 633  862]]
Confusion matrix (Test):
[[938  97]
 [181 193]]
